In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up paths
ROOT_DIR = Path().resolve()
while not (ROOT_DIR / 'studies').exists():
    ROOT_DIR = ROOT_DIR.parent

trips_file = ROOT_DIR / 'data' / 'demand' / 'trips_klagenfurt.csv'
baseline_file = ROOT_DIR / 'data' / 'demand' / 'klagenfurt_h1_spitzenzeiten' / 'matched' / 'klagenfurt_h1_spitzenzeiten' / 'demand_requests.csv'
network_nodes = ROOT_DIR / 'data' / 'networks' / 'klagenfurt_h1_spitzenzeiten' / 'base' / 'nodes.csv'

print(f"Root directory: {ROOT_DIR}")
print(f"Trips file exists: {trips_file.exists()}")
print(f"Baseline file exists: {baseline_file.exists()}")
print(f"Network nodes file exists: {network_nodes.exists()}")

# MOIA Trips Dataset Analysis
## Exploratory Data Analysis for trips_klagenfurt.csv

This notebook analyzes the new MOIA demand data (trips_klagenfurt.csv) and compares it with the existing klagenfurt_h1_spitzenzeiten baseline to assess viability for rerun with `--track trips`.

## 1. Load and Inspect Trips Dataset

In [ ]:
# Load MOIA trips data
trips = pd.read_csv(trips_file)
print(f"Shape: {trips.shape}")
print(f"\nColumns: {trips.columns.tolist()}")
print(f"\nData types:\n{trips.dtypes}")
print(f"\nFirst 5 rows:")
trips.head()

In [ ]:
# Check for missing values and basic statistics
print("Missing values:")
print(trips.isnull().sum())
print(f"\nBasic statistics:")
print(trips.describe())

## 2. Temporal Analysis

In [ ]:
# Parse departure times (assuming seconds since midnight or epoch)
trips['departure_time_sec'] = trips['departure_time']

# Convert to hours and minutes for visualization
trips['departure_hour'] = trips['departure_time_sec'] // 3600
trips['departure_minute'] = (trips['departure_time_sec'] % 3600) // 60
trips['time_of_day'] = trips['departure_hour'] + trips['departure_minute'] / 60

# Apply modulo 86400 to handle multi-day data
trips['seconds_of_day'] = trips['departure_time_sec'] % 86400
trips['hour_of_day'] = trips['seconds_of_day'] // 3600

print(f"Departure time range (raw): {trips['departure_time_sec'].min()} to {trips['departure_time_sec'].max()}")
print(f"Seconds of day range: {trips['seconds_of_day'].min()} to {trips['seconds_of_day'].max()}")
print(f"Hours of day range: {trips['hour_of_day'].min()} to {trips['hour_of_day'].max()}")
print(f"\nDeparture hour distribution:")
print(trips['hour_of_day'].value_counts().sort_index())

In [ ]:
# Temporal distribution visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Histogram of hour of day
ax = axes[0]
trips['hour_of_day'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Trips')
ax.set_title('MOIA Trips: Demand by Hour of Day')
ax.grid(axis='y', alpha=0.3)

# Cumulative distribution
ax = axes[1]
cumsum = trips['hour_of_day'].value_counts().sort_index().cumsum()
ax.plot(cumsum.index, cumsum.values, marker='o', linewidth=2, markersize=4, color='darkblue')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Cumulative Trips')
ax.set_title('Cumulative Demand by Hour')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT_DIR / 'data' / 'demand' / 'klagenfurt_h1_spitzenzeiten' / 'moia_temporal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Peak hour: {trips['hour_of_day'].value_counts().idxmax()} with {trips['hour_of_day'].value_counts().max()} trips")

## 3. Spatial Analysis

In [ ]:
# Load network nodes to determine spatial coverage
nodes = pd.read_csv(network_nodes)
print(f"Network nodes: {len(nodes)} nodes")
print(f"Network bounds: x=[{nodes['pos_x'].min():.6f}, {nodes['pos_x'].max():.6f}], y=[{nodes['pos_y'].min():.6f}, {nodes['pos_y'].max():.6f}]")

# Determine spatial coverage of MOIA trips
origin_in_bbox = (trips['origin_x'].between(nodes['pos_x'].min(), nodes['pos_x'].max()) & 
                   trips['origin_y'].between(nodes['pos_y'].min(), nodes['pos_y'].max()))
destination_in_bbox = (trips['destination_x'].between(nodes['pos_x'].min(), nodes['pos_x'].max()) & 
                       trips['destination_y'].between(nodes['pos_y'].min(), nodes['pos_y'].max()))

both_in_bbox = origin_in_bbox & destination_in_bbox

print(f"\nSpatial coverage:")
print(f"Origins in network bbox: {origin_in_bbox.sum()}/{len(trips)} ({100*origin_in_bbox.sum()/len(trips):.1f}%)")
print(f"Destinations in network bbox: {destination_in_bbox.sum()}/{len(trips)} ({100*destination_in_bbox.sum()/len(trips):.1f}%)")
print(f"Both endpoints in bbox: {both_in_bbox.sum()}/{len(trips)} ({100*both_in_bbox.sum()/len(trips):.1f}%)")

# Trips MOIA bounds
print(f"\nMOIA trips bounds:")
print(f"Origin x: [{trips['origin_x'].min():.6f}, {trips['origin_x'].max():.6f}]")
print(f"Origin y: [{trips['origin_y'].min():.6f}, {trips['origin_y'].max():.6f}]")
print(f"Destination x: [{trips['destination_x'].min():.6f}, {trips['destination_x'].max():.6f}]")
print(f"Destination y: [{trips['destination_y'].min():.6f}, {trips['destination_y'].max():.6f}]")

In [ ]:
# Spatial distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Origins scatter plot
ax = axes[0]
ax.scatter(trips['origin_x'], trips['origin_y'], alpha=0.5, s=20, color='blue', label='Origins')
ax.scatter(nodes['pos_x'], nodes['pos_y'], alpha=0.3, s=5, color='red', label='Network nodes')
ax.set_xlabel('X Coordinate (°)')
ax.set_ylabel('Y Coordinate (°)')
ax.set_title('MOIA Trip Origins vs Network Coverage')
ax.legend()
ax.grid(alpha=0.3)

# Destinations scatter plot
ax = axes[1]
ax.scatter(trips['destination_x'], trips['destination_y'], alpha=0.5, s=20, color='green', label='Destinations')
ax.scatter(nodes['pos_x'], nodes['pos_y'], alpha=0.3, s=5, color='red', label='Network nodes')
ax.set_xlabel('X Coordinate (°)')
ax.set_ylabel('Y Coordinate (°)')
ax.set_title('MOIA Trip Destinations vs Network Coverage')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT_DIR / 'data' / 'demand' / 'klagenfurt_h1_spitzenzeiten' / 'moia_spatial_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Comparison with Baseline Demand

In [ ]:
# Load baseline demand
baseline = pd.read_csv(baseline_file)
print(f"Baseline demand shape: {baseline.shape}")
print(f"Baseline columns: {baseline.columns.tolist()}")
print(f"Baseline first rows:")
baseline.head()

# Parse baseline times
baseline['hour_of_day'] = baseline['rq_time'] // 3600
print(f"\nBaseline temporal range: {baseline['rq_time'].min()} to {baseline['rq_time'].max()} seconds")
print(f"Baseline hour distribution:")
print(baseline['hour_of_day'].value_counts().sort_index())

In [ ]:
# Comparison: Size and temporal characteristics
comparison_data = {
    'Dataset': ['MOIA Trips', 'Baseline'],
    'Total Requests': [len(trips), len(baseline)],
    'Hour Range (min)': [trips['hour_of_day'].min(), baseline['hour_of_day'].min()],
    'Hour Range (max)': [trips['hour_of_day'].max(), baseline['hour_of_day'].max()],
    'Peak Hour': [trips['hour_of_day'].value_counts().idxmax(), baseline['hour_of_day'].value_counts().idxmax()],
    'Peak Hour Count': [trips['hour_of_day'].value_counts().max(), baseline['hour_of_day'].value_counts().max()],
}

comparison_df = pd.DataFrame(comparison_data)
print("Comparison Summary:")
print(comparison_df)

# Temporal alignment
fig, ax = plt.subplots(figsize=(12, 5))
trips_hourly = trips['hour_of_day'].value_counts().sort_index()
baseline_hourly = baseline['hour_of_day'].value_counts().sort_index()

ax.plot(trips_hourly.index, trips_hourly.values, marker='o', label='MOIA Trips', linewidth=2, markersize=6)
ax.plot(baseline_hourly.index, baseline_hourly.values, marker='s', label='Baseline', linewidth=2, markersize=6)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Requests')
ax.set_title('Demand Distribution: MOIA vs Baseline')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT_DIR / 'data' / 'demand' / 'klagenfurt_h1_spitzenzeiten' / 'moia_vs_baseline_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Trip Distance Analysis

In [ ]:
# Calculate trip distances (Euclidean)
trips['distance_x'] = trips['destination_x'] - trips['origin_x']
trips['distance_y'] = trips['destination_y'] - trips['origin_y']
trips['distance_euclidean'] = np.sqrt(trips['distance_x']**2 + trips['distance_y']**2)

print(f"Trip distance statistics (degrees):")
print(trips['distance_euclidean'].describe())
print(f"\nTrips with distance < 0.01 degrees: {(trips['distance_euclidean'] < 0.01).sum()} ({100*(trips['distance_euclidean'] < 0.01).sum()/len(trips):.1f}%)")
print(f"Trips with distance < 0.001 degrees: {(trips['distance_euclidean'] < 0.001).sum()} ({100*(trips['distance_euclidean'] < 0.001).sum()/len(trips):.1f}%)")

# Distance distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(trips['distance_euclidean'], bins=50, edgecolor='black', color='steelblue')
ax.set_xlabel('Euclidean Distance (degrees)')
ax.set_ylabel('Number of Trips')
ax.set_title('MOIA Trip Distance Distribution')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT_DIR / 'data' / 'demand' / 'klagenfurt_h1_spitzenzeiten' / 'moia_distance_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Data Quality Assessment

In [ ]:
print("=== DATA QUALITY ASSESSMENT ===\n")

# Check for duplicates
exact_duplicates = trips.duplicated().sum()
print(f"Exact duplicates: {exact_duplicates} ({100*exact_duplicates/len(trips):.2f}%)")

# Check for same origin/destination
same_location = (trips['distance_euclidean'] < 1e-6).sum()
print(f"Same origin/destination: {same_location} ({100*same_location/len(trips):.2f}%)")

# Check for missing values
missing = trips[['departure_time', 'origin_x', 'origin_y', 'destination_x', 'destination_y']].isnull().sum()
print(f"\nMissing values:")
print(missing)

# Spatial coverage assessment
print(f"\n=== SPATIAL COVERAGE ASSESSMENT ===")
print(f"Trips with both endpoints in network bbox: {both_in_bbox.sum()}/{len(trips)} ({100*both_in_bbox.sum()/len(trips):.1f}%)")
print(f"Usable trips (can be snapped to network): ~{both_in_bbox.sum()} out of {len(trips)}")
print(f"Projected coverage: {100*both_in_bbox.sum()/len(trips):.1f}%")

# Temporal coverage assessment
print(f"\n=== TEMPORAL COVERAGE ASSESSMENT ===")
print(f"Time window: {int(trips['seconds_of_day'].min())} to {int(trips['seconds_of_day'].max())} seconds of day")
print(f"In standard peak window (6:00-21:00 / 21600-75600s): {((trips['seconds_of_day'] >= 21600) & (trips['seconds_of_day'] <= 75600)).sum()} trips")
print(f"In H1 morning peak (6:00-9:00 / 21600-32400s): {((trips['seconds_of_day'] >= 21600) & (trips['seconds_of_day'] <= 32400)).sum()} trips")

## 7. Summary and Recommendations

In [ ]:
print("=== SUMMARY ===\n")

summary = f"""
MOIA Trips Dataset Summary:
- Total records: {len(trips)}
- Spatial coverage: {100*both_in_bbox.sum()/len(trips):.1f}% of trips have both endpoints in network bbox
- Temporal range: Hours {trips['hour_of_day'].min()}-{trips['hour_of_day'].max()} of day
- Peak hour: {trips['hour_of_day'].value_counts().idxmax()}:00 with {trips['hour_of_day'].value_counts().max()} trips
- Average trip distance: {trips['distance_euclidean'].mean():.6f}° (Euclidean)
- Data quality: No exact duplicates, {both_in_bbox.sum()} usable trips

Comparison with Baseline:
- Baseline size: {len(baseline)} requests (vs {len(trips)} MOIA trips)
- Baseline temporal range: Hours {baseline['hour_of_day'].min()}-{baseline['hour_of_day'].max()}
- Scale difference: {len(trips)/len(baseline):.1f}x MOIA vs baseline

Conversion Status:
- Converter script: convert_trips_klagenfurt_to_requests.py
- Next step: Run converter to generate demand_requests_trips.csv
- Expected usable records: ~{both_in_bbox.sum()} (after nearest-node snapping and time filtering)

For Rerun:
- Use: python run_klagenfurt_h1_spitzenzeiten.py --track trips
- Expected output directory: studies/klagenfurt_h1_spitzenzeiten_case_study/results/*__trips/
"""

print(summary)